# Forest Memory — Gemma Ecological Reasoning

**Hackathon:** Kaggle Gemma 4 Good

This notebook loads `outputs/forest_memory_cases.json` and uses **Gemma** to generate
structured ecological resilience reports from multimodal environmental signals.

---

### What Gemma does here

Gemma is **not** a classifier, detector, or prediction engine.

Gemma acts as an **ecological reasoning engine**: it receives NDVI satellite data,
bioacoustic proxy scores, fire-history metadata, and uncertainty flags, then synthesises
a structured resilience report with careful proxy language.

---

> **Scientific disclaimer:** All acoustic and spectral-index values are proxy signals only.
> Reports do not represent confirmed species counts, validated biodiversity measurements,
> or wildfire predictions. Values are relative within this 4-site sample.

## 0  Setup

In [1]:
import json, os, sys, time, textwrap
from pathlib import Path
from IPython.display import display, Markdown

# Kaggle path resolution
DATASET_ROOT = Path("/kaggle/input/datasets/zeynepucuncuoglu/forest-memory-data/forest-memory")
if not DATASET_ROOT.exists():
    DATASET_ROOT = Path("..").resolve()  # local run

CASES_JSON = DATASET_ROOT / "outputs" / "forest_memory_cases.json"
OUT_DIR    = Path("/kaggle/working/gemma_reports")
OUT_JSON   = OUT_DIR / "gemma_reports.json"
OUT_PRETTY = OUT_DIR / "gemma_reports_pretty.json"
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Cases JSON:", CASES_JSON.exists(), CASES_JSON)
print("Output dir:", OUT_DIR)


Cases JSON: True /kaggle/input/datasets/zeynepucuncuoglu/forest-memory-data/forest-memory/outputs/forest_memory_cases.json
Output dir: /kaggle/working/gemma_reports


## 1  Configure Gemma API

Supports three authentication paths — whichever is available first is used:
1. **Kaggle Secrets** (`GOOGLE_API_KEY`) — preferred on Kaggle
2. **Environment variable** `GOOGLE_API_KEY`
3. **Manual entry** via `getpass`

In [2]:
import google.genai as genai

api_key = None
try:
    from kaggle_secrets import UserSecretsClient
    api_key = UserSecretsClient().get_secret("GOOGLE_API_KEY")
    print("API key loaded from Kaggle Secrets.")
except Exception:
    pass

if not api_key:
    api_key = os.environ.get("GOOGLE_API_KEY")
    if api_key:
        print("API key loaded from environment variable.")

if not api_key:
    import getpass
    api_key = getpass.getpass("Enter your Google AI API key: ")

client = genai.Client(api_key=api_key)
MODEL_NAME = "gemma-4-31b-it"
print(f"Model: {MODEL_NAME}")


API key loaded from Kaggle Secrets.
Model: gemma-4-31b-it


## 2  Load Forest Memory cases

In [3]:
with open(CASES_JSON) as f:
    cases = json.load(f)

print(f"{len(cases)} cases loaded")
for c in cases:
    flags = [k for k, v in c["interpretation_flags"].items() if v]
    print(f"  {c['role']:30s}  site={c['site_id']}  flags={flags or ['none']}")

4 cases loaded
  healthy_baseline                site=s2lam027_231127  flags=['none']
  burned_recovering               site=s2lam047_231107  flags=['green_not_alive_signal', 'spatially_variable_vegetation', 'recent_fire_recovery_context']
  invasive_disturbed              site=s2lam051_231030  flags=['green_not_alive_signal']
  wet_dry_pair_complement         site=s2lam096_230823  flags=['recent_fire_recovery_context', 'acoustic_uncertainty']


## 3  Prompt construction

In [4]:
def format_case_for_prompt(case: dict) -> str:
    """Format a Forest Memory case dict into readable text for Gemma."""
    meta  = case.get("site_metadata", {})
    audio = case.get("audio", {})
    ndvi  = case.get("ndvi") or {}
    flags = case.get("interpretation_flags", {})

    def _s(v, fallback="not recorded"):
        return str(v) if v is not None else fallback

    def _score(col):
        d = audio.get(col, {})
        if not d or d.get("mean") is None:
            return "not available"
        return f"{d['mean']:.1f} / 100  (min {d['min']:.1f}, max {d['max']:.1f})"

    lines = [
        f"SITE ROLE:  {case['role'].replace('_', ' ').upper()}",
        f"SITE ID:    {case['site_id']}",
        "",
        "─── ECOSYSTEM CONTEXT ───────────────────────────────────────",
        f"  Land cover class :  {_s(meta.get('land_cover_class'))}",
        f"  Fire history class:  {_s(meta.get('fire_class'))}",
        f"  Vegetation age   :  {_s(meta.get('field_veld_age'))}",
        f"  Invasive pressure:  {_s(meta.get('field_aliens_within_20m'))}",
        f"  Elevation class  :  {_s(meta.get('elevation_class'))}",
        f"  Campaign season  :  {_s(meta.get('campaign'))}",
        "",
        "─── SATELLITE VEGETATION — NDVI (Sentinel-2, proxy) ─────────",
    ]

    if ndvi:
        lines += [
            f"  Mean NDVI        :  {ndvi['mean_ndvi']:.4f}  (0 = bare soil, 1 = dense green)",
            f"  NDVI variability :  std = {ndvi['std_ndvi']:.4f}  "
            f"(range {ndvi['min_ndvi']:.3f} – {ndvi['max_ndvi']:.3f})",
            f"  Images analysed  :  {ndvi['image_count']} (period {ndvi['period_start']} to {ndvi['period_end']})",
        ]
    else:
        lines.append("  NDVI data: not available for this site")

    lines += [
        "",
        "─── ACOUSTIC PROXIES (passive monitoring, 60 s WAV clips) ───",
        f"  WAV recordings           :  {audio.get('wav_file_count', '?')} files analysed",
        f"  Bioacoustic vitality     :  {_score('bioacoustic_vitality_score')}",
        f"  Acoustic richness        :  {_score('acoustic_richness_score')}",
        f"  Bird activity (proxy)    :  {_score('bird_activity_proxy')}",
        f"  Insect activity (proxy)  :  {_score('insect_activity_proxy_score')}",
        f"  Human noise interference :  {_score('human_disturbance_proxy')}",
        "",
        "─── INTERPRETATION FLAGS ────────────────────────────────────",
        f"  green_not_alive_signal       :  {flags.get('green_not_alive_signal', False)}",
        f"    (NDVI high but vitality low — possible mismatch between greenness and acoustic life)",
        f"  spatially_variable_vegetation:  {flags.get('spatially_variable_vegetation', False)}",
        f"    (high NDVI temporal std — patchy or heterogeneous vegetation)",
        f"  recent_fire_recovery_context :  {flags.get('recent_fire_recovery_context', False)}",
        f"    (fire history metadata indicates 1-to-6 year post-fire window)",
        f"  acoustic_uncertainty         :  {flags.get('acoustic_uncertainty', False)}",
        f"    (high low-frequency energy — wind/noise may mask biological signals)",
    ]

    return "\n".join(lines)


SITE_PROMPT_TEMPLATE = """\
Analyse the following ecosystem site data and produce an ecological resilience report.

{site_data}

Write a structured report with exactly these five sections.
Keep each section to 2–4 sentences.
Use careful proxy language throughout — never overclaim.

## Vegetation Interpretation
(What does the NDVI signal suggest about vegetation state? Note any tension between
 greenness and other signals, or flag missing data.)

## Bioacoustic Interpretation
(What do the acoustic proxy scores suggest about soundscape activity?
 Note dominant signals and any high uncertainty.)

## Multimodal Tension Summary
(Where do satellite and acoustic signals agree or diverge? What might this divergence —
 or alignment — indicate about the ecosystem state?)

## Recovery Interpretation
(Given fire history, land cover, and combined proxy data, what recovery trajectory
 does this site appear to be on? Be explicit about what is unknown.)

## Uncertainty Notes
(What data limitations, missing signals, seasonal confounders, or methodological
 caveats should a practitioner consider when interpreting this report?)
"""

# Quick preview of one formatted case
print(format_case_for_prompt(cases[0]))

SITE ROLE:  HEALTHY BASELINE
SITE ID:    s2lam027_231127

─── ECOSYSTEM CONTEXT ───────────────────────────────────────
  Land cover class :  Shrubland-Fynbos
  Fire history class:  No data or No Fire
  Vegetation age   :  Old (> 17 yrs)
  Invasive pressure:  not recorded
  Elevation class  :  Low: 0-500 m
  Campaign season  :  Dry season

─── SATELLITE VEGETATION — NDVI (Sentinel-2, proxy) ─────────
  Mean NDVI        :  0.5071  (0 = bare soil, 1 = dense green)
  NDVI variability :  std = 0.0578  (range 0.256 – 0.746)
  Images analysed  :  39 (period 2023-10-01 to 2023-12-20)

─── ACOUSTIC PROXIES (passive monitoring, 60 s WAV clips) ───
  WAV recordings           :  3 files analysed
  Bioacoustic vitality     :  41.1 / 100  (min 26.9, max 57.8)
  Acoustic richness        :  51.4 / 100  (min 46.6, max 53.9)
  Bird activity (proxy)    :  56.1 / 100  (min 48.7, max 66.2)
  Insect activity (proxy)  :  57.2 / 100  (min 30.6, max 91.4)
  Human noise interference :  27.8 / 100  (min 20.1, m

## 4  Generate per-site reports


In [5]:
def parse_sections(text: str) -> dict[str, str]:
    section_keys = [
        "Vegetation Interpretation",
        "Bioacoustic Interpretation",
        "Multimodal Tension Summary",
        "Recovery Interpretation",
        "Uncertainty Notes",
    ]
    sections: dict[str, str] = {}
    current_key = None
    buffer: list[str] = []
    for line in text.splitlines():
        stripped = line.strip()
        matched = next((k for k in section_keys if stripped.startswith(f"## {k}")), None)
        if matched:
            if current_key and buffer:
                sections[current_key] = " ".join(" ".join(buffer).split())
            current_key = matched
            buffer = []
        elif current_key and stripped and not stripped.startswith("#"):
            buffer.append(stripped)
    if current_key and buffer:
        sections[current_key] = " ".join(" ".join(buffer).split())
    for k in section_keys:
        sections.setdefault(k, "(not generated)")
    return sections


def generate_site_report(case: dict) -> dict:
    prompt = SITE_PROMPT_TEMPLATE.format(site_data=format_case_for_prompt(case))
    t0 = time.time()
    raw_text = "[Error]"
    for attempt in range(3):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=genai.types.GenerateContentConfig(
                    temperature=0.3, max_output_tokens=800
                ),
            )
            raw_text = response.text
            break
        except Exception as e:
            print(f"  attempt {attempt+1} failed: {e}")
            time.sleep(5)
    return {
        "role":             case["role"],
        "site_id":          case["site_id"],
        "model":            MODEL_NAME,
        "sections":         parse_sections(raw_text),
        "raw_response":     raw_text,
        "proxy_disclaimer": case.get("proxy_disclaimer", ""),
        "latency_s":        round(time.time() - t0, 2),
    }


print("Generating reports...")
reports = []
for i, case in enumerate(cases):
    print(f"  [{i+1}/{len(cases)}] {case['role']} ...", end=" ", flush=True)
    report = generate_site_report(case)
    reports.append(report)
    print(f"{report['latency_s']}s")
    if i < len(cases) - 1:
        time.sleep(3)
print(f"\n{len(reports)} reports generated.")


Generating reports...
  [1/4] healthy_baseline ... 34.45s
  [2/4] burned_recovering ... 33.08s
  [3/4] invasive_disturbed ... 29.58s
  [4/4] wet_dry_pair_complement ... 29.46s

4 reports generated.


## 5  Save reports

In [6]:
with open(OUT_JSON, "w") as f:
    json.dump(reports, f, separators=(",", ":"))

with open(OUT_PRETTY, "w") as f:
    json.dump(reports, f, indent=2)

print(f"Saved -> {OUT_JSON}")
print(f"Saved -> {OUT_PRETTY}")


Saved -> /kaggle/working/gemma_reports/gemma_reports.json
Saved -> /kaggle/working/gemma_reports/gemma_reports_pretty.json


## 6  Display reports

In [7]:
ROLE_EMOJI = {
    "healthy_baseline":       "🌿",
    "burned_recovering":      "🔥",
    "invasive_disturbed":     "⚠️",
    "wet_dry_pair_complement": "💧",
}

SECTION_ORDER = [
    "Vegetation Interpretation",
    "Bioacoustic Interpretation",
    "Multimodal Tension Summary",
    "Recovery Interpretation",
    "Uncertainty Notes",
]

for report in reports:
    emoji = ROLE_EMOJI.get(report["role"], "📍")
    role_display = report["role"].replace("_", " ").title()

    md_parts = [
        f"---",
        f"## {emoji} {role_display}  ",
        f"**Site ID:** `{report['site_id']}`  |  **Model:** `{report['model']}`",
        "",
    ]
    for section in SECTION_ORDER:
        text = report["sections"].get(section, "(not generated)")
        md_parts.append(f"### {section}")
        md_parts.append(text)
        md_parts.append("")

    md_parts.append(
        f"> *{report['proxy_disclaimer']}*"
    )
    display(Markdown("\n".join(md_parts)))

---
## 🌿 Healthy Baseline  
**Site ID:** `s2lam027_231127`  |  **Model:** `gemma-4-31b-it`

### Vegetation Interpretation
The mean NDVI of 0.5071 suggests a moderate level of greenness, which is notable given the dry season context. Low temporal variability indicates a relatively stable vegetation cover across the analyzed period. These signals are consistent with the reported old-growth shrubland status.

### Bioacoustic Interpretation
Acoustic proxy scores indicate moderate levels of bioacoustic vitality (41.1) and richness (51.4). Bird and insect activity proxies are slightly higher, suggesting a present but not dominant faunal soundscape. Human noise interference remains relatively low.

### Multimodal Tension Summary
There is a general alignment between the moderate vegetation greenness and the moderate acoustic activity. The absence of a "green_not_alive" flag suggests that the biomass levels are reasonably supported by biological activity. No significant divergence between satellite and acoustic proxies was detected.

### Recovery Interpretation
With a vegetation age exceeding 17 years and no recorded recent fire, the site does not appear to be in an active recovery phase. It likely represents a late-successional state, though the absence of specific fire history data limits a definitive trajectory analysis. The site currently functions as a baseline for old-growth Fynbos.

### Uncertainty Notes
The bioacoustic analysis is based on a very small sample size of only three WAV files, which may not capture full diurnal or spatial variation. Dry season timing may also confound activity levels for certain species. Additionally, the lack of invasive pressure data prevents an assessment of biotic threats.

> *All acoustic and NDVI values are proxy signals only. Do not interpret as confirmed species counts, validated biodiversity measurements, or wildfire predictions. Values are relative within this sample of sites and recording sessions.*

---
## 🔥 Burned Recovering  
**Site ID:** `s2lam047_231107`  |  **Model:** `gemma-4-31b-it`

### Vegetation Interpretation
The mean NDVI of 0.5077 suggests moderate greenness, though high variability indicates a patchy or heterogeneous vegetation structure. This signal is consistent with a site in early-to-mid recovery following a fire event. The data suggests a non-uniform return of biomass across the site.

### Bioacoustic Interpretation
Acoustic proxies suggest moderate richness but relatively low overall bioacoustic vitality. Insect activity appears more prominent than bird activity, while human noise interference remains low. These scores indicate a soundscape with limited biological intensity.

### Multimodal Tension Summary
There is a notable divergence between the moderate NDVI signal and the low bioacoustic vitality score. This tension suggests that while vegetation is returning, the associated faunal community may not yet have fully recolonized the site. The "greenness" of the site is not currently mirrored by high acoustic activity.

### Recovery Interpretation
The site appears to be in an early-successional recovery phase, where primary productivity precedes the return of complex animal assemblages. Low invasive pressure is a positive indicator for the trajectory of the Fynbos shrubland. It remains unknown if the low faunal activity is a permanent state or a temporary lag in the recovery timeline.

### Uncertainty Notes
The analysis is limited by a very small acoustic sample size of only three recordings. Additionally, the dry season timing of the campaign may suppress biological activity, potentially confounding the vitality and richness scores. NDVI remains a proxy for greenness and does not provide specific data on species composition.

> *All acoustic and NDVI values are proxy signals only. Do not interpret as confirmed species counts, validated biodiversity measurements, or wildfire predictions. Values are relative within this sample of sites and recording sessions.*

---
## ⚠️ Invasive Disturbed  
**Site ID:** `s2lam051_231030`  |  **Model:** `gemma-4-31b-it`

### Vegetation Interpretation
The mean NDVI of 0.6774 suggests a moderately dense green canopy, typical of forest-woodland cover. However, the presence of a `green_not_alive_signal` flag indicates a potential mismatch between this spectral greenness and actual biological vitality. Temporal variability in the NDVI suggests some fluctuations in vegetation state during the dry season.

### Bioacoustic Interpretation
Acoustic proxies indicate low overall bioacoustic vitality (31.1), despite moderate scores for richness and bird activity. Insect activity appears low and highly inconsistent across the sampled clips. Human noise interference is present but remains at a relatively low level.

### Multimodal Tension Summary
A significant divergence exists between the high satellite-derived greenness and the low bioacoustic vitality. This tension suggests that the vegetatively productive state indicated by the NDVI may not be supporting a proportional level of faunal activity. Such a gap is often associated with sites where invasive species dominate the biomass.

### Recovery Interpretation
The site is in an intermediate post-fire stage (6–12 years) and has successfully established canopy cover. However, the combined proxy data suggests the recovery trajectory may be skewed toward an invasive-dominated state rather than a diverse native ecosystem. It remains unknown whether this current state is stable or if faunal recruitment is lagging behind vegetative growth.

### Uncertainty Notes
The acoustic interpretations are based on a very limited sample size of three files, which may not represent the full spatial or temporal complexity of the site. Data collection occurred during the dry season, which can naturally suppress insect and bird activity proxies. Additionally, the "Invasive Disturbed" site role suggests that NDVI may be reflecting invasive biomass rather than native forest health.

> *All acoustic and NDVI values are proxy signals only. Do not interpret as confirmed species counts, validated biodiversity measurements, or wildfire predictions. Values are relative within this sample of sites and recording sessions.*

---
## 💧 Wet Dry Pair Complement  
**Site ID:** `s2lam096_230823`  |  **Model:** `gemma-4-31b-it`

### Vegetation Interpretation
NDVI data is unavailable for this site, preventing a direct assessment of current greenness or biomass. Consequently, the vegetation state must be inferred solely from the Shrubland-Fynbos land cover classification and fire history metadata.

### Bioacoustic Interpretation
Acoustic proxies suggest low overall bioacoustic vitality, despite moderate richness and bird activity scores. High human noise interference and the presence of low-frequency energy suggest that biological signals may be partially masked, increasing the uncertainty of these metrics.

### Multimodal Tension Summary
A full multimodal comparison is constrained by the absence of satellite vegetation data. However, the divergence between moderate acoustic richness and low bioacoustic vitality may indicate a soundscape where a few species are present but overall biological intensity remains low.

### Recovery Interpretation
The site is within a 1-to-6 year post-fire recovery window, which is a critical phase for Fynbos regeneration. It is currently unclear whether the low bioacoustic vitality reflects an early stage of ecological succession or is an artifact of high anthropogenic noise.

### Uncertainty Notes
The lack of NDVI data and the limited sample size of two audio files reduce the statistical confidence of this report. Practitioners should also consider that high human noise and potential wind interference may have suppressed the detected biological proxies.

> *All acoustic and NDVI values are proxy signals only. Do not interpret as confirmed species counts, validated biodiversity measurements, or wildfire predictions. Values are relative within this sample of sites and recording sessions.*

## 7  Cross-site synthesis


In [8]:
def _condensed(case: dict) -> str:
    meta  = case["site_metadata"]
    audio = case["audio"]
    ndvi  = case.get("ndvi") or {}
    flags = [k for k, v in case["interpretation_flags"].items() if v]
    vitality = (audio.get("bioacoustic_vitality_score") or {}).get("mean")
    ndvi_m   = ndvi.get("mean_ndvi")
    ndvi_s   = ndvi.get("std_ndvi")
    return (
        f"[{case['role']}]  "
        f"Land cover: {meta.get('land_cover_class', '?')}  "
        f"Fire: {meta.get('fire_class', '?')}  "
        f"Vitality proxy: {f'{vitality:.1f}/100' if vitality is not None else 'N/A'}  "
        f"NDVI mean: {f'{ndvi_m:.3f}' if ndvi_m else 'N/A'}  "
        f"Flags: {', '.join(flags) if flags else 'none'}"
    )

cross_site_data = "\n".join(_condensed(c) for c in cases)

synthesis_prompt = f"""\
You have analysed four ecosystem monitoring sites from the BioSCape campaign
in the Cape Floristic Region, South Africa. Here is a summary of all four:

{cross_site_data}

Write a cross-site ecological resilience synthesis with these three sections.
2-4 sentences per section. Use careful proxy language throughout.

## Comparative Resilience Narrative
## Key Multimodal Insight
## Conservation Monitoring Implications
"""

print("Generating cross-site synthesis...")
t0 = time.time()
synthesis_text = "[Error]"
for attempt in range(3):
    try:
        synthesis_response = client.models.generate_content(
            model=MODEL_NAME,
            contents=synthesis_prompt,
            config=genai.types.GenerateContentConfig(
                temperature=0.3, max_output_tokens=1000
            ),
        )
        synthesis_text = synthesis_response.text
        print(f"Done ({time.time() - t0:.1f}s)")
        break
    except Exception as e:
        print(f"  attempt {attempt+1} failed: {e}")
        time.sleep(5)

all_outputs = {"site_reports": reports, "synthesis": {"model": MODEL_NAME, "raw_response": synthesis_text}}
with open(OUT_PRETTY, "w") as f:
    json.dump(all_outputs, f, indent=2)
with open(OUT_JSON, "w") as f:
    json.dump(all_outputs, f, separators=(",", ":"))
print(f"Saved -> {OUT_PRETTY}")


Generating cross-site synthesis...
Done (31.9s)
Saved -> /kaggle/working/gemma_reports/gemma_reports_pretty.json


In [9]:
SYNTHESIS_SECTIONS = [
    "Comparative Resilience Narrative",
    "Key Multimodal Insight",
    "Conservation Monitoring Implications",
]

synthesis_sections = parse_sections(synthesis_text)
# parse_sections looks for "## X" — map the synthesis-specific keys

md_parts = [
    "---",
    "## 🌍 Cross-Site Ecological Synthesis",
    f"**Model:** `{MODEL_NAME}`",
    "",
]
for section in SYNTHESIS_SECTIONS:
    text = synthesis_sections.get(section, "")
    if not text:
        # fallback: print raw if sections didn't parse cleanly
        md_parts += ["### Raw synthesis output", synthesis_text]
        break
    md_parts.append(f"### {section}")
    md_parts.append(text)
    md_parts.append("")

md_parts.append(
    "> *All values are acoustic and spectral-index proxies — "
    "not confirmed biodiversity measurements.*"
)
display(Markdown("\n".join(md_parts)))

---
## 🌍 Cross-Site Ecological Synthesis
**Model:** `gemma-4-31b-it`

### Raw synthesis output
## Comparative Resilience Narrative
The `healthy_baseline` site exhibits the highest vitality proxy, serving as a potential reference for stability within the shrubland-fynbos land cover. In contrast, the `burned_recovering` and `wet_dry_pair_complement` sites show divergent vitality proxy levels following recent fire events, suggesting varied trajectories of post-disturbance recovery. The `invasive_disturbed` site demonstrates a notable decoupling, where a high NDVI mean coincides with a lower vitality proxy, potentially indicating a shift in ecosystem composition.

## Key Multimodal Insight
A critical discrepancy exists where spectral greenness does not consistently align with vitality proxy scores, particularly evidenced by the "green_not_alive" signals in the recovering and disturbed sites. This suggests that high NDVI may mask underlying ecological stress or the presence of invasive species rather than indicating true ecosystem vigor. Consequently, relying on a single remote sensing metric may lead to an overestimation of resilience in degraded or transitioning landscapes.

## Conservation Monitoring Implications
Monitoring frameworks should prioritize the integration of multimodal indicators to better differentiate between natural successional recovery and invasive encroachment. Addressing acoustic uncertainty and spatial variability will be essential for refining the accuracy of vitality proxies across complex fynbos mosaics. Such a nuanced approach allows practitioners to more precisely identify sites requiring urgent intervention versus those following expected post-fire recovery paths.
> *All values are acoustic and spectral-index proxies — not confirmed biodiversity measurements.*

## 8  Quick reference — report quality check

In [10]:
# Flag any reports that are suspiciously short (possible API error or truncation)
MIN_CHARS = 150
print("Report quality check:")
print(f"{'Role':32s}  {'Sections OK':12s}  {'Min section len':16s}  Status")
print("-" * 80)
for r in reports:
    secs = r["sections"]
    n_ok = sum(1 for v in secs.values() if len(v) >= MIN_CHARS)
    min_len = min(len(v) for v in secs.values())
    status = "✅" if n_ok == 5 else f"⚠️  only {n_ok}/5 sections ≥{MIN_CHARS} chars"
    print(f"{r['role']:32s}  {n_ok}/5         {min_len:6d} chars     {status}")

Report quality check:
Role                              Sections OK   Min section len   Status
--------------------------------------------------------------------------------
healthy_baseline                  5/5            258 chars     ✅
burned_recovering                 5/5            264 chars     ✅
invasive_disturbed                5/5            275 chars     ✅
wet_dry_pair_complement           5/5            241 chars     ✅


---

## Output summary

| File | Contents |
|---|---|
| `outputs/gemma_reports/gemma_reports.json` | Compact JSON: all site reports + synthesis |
| `outputs/gemma_reports/gemma_reports_pretty.json` | Human-readable version |

---

**Next steps in the Forest Memory pipeline:**
- `04_litert_edge.ipynb` — LiteRT model quantisation for edge deployment
- `05_gradio_demo.py` — Interactive Gradio UI wiring reports → display
- Demo video production